# MoE Mutation 001 — formal Kaggle launcher

This notebook is an execution launcher only. Scientific logic and thresholds are frozen in `research/validations/moe-mutation-001/protocol.json`; training is implemented in `scripts/research/moe_mutation_001/run_seed.py`. Each completed seed is published immediately before the next seed begins.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPOSITORY = 'https://github.com/ArcheLabs/mini-cells.git'
BRANCH = 'codex/moe-mutation-001-kaggle'
ROOT = Path('/kaggle/working/mini-cells')

if ROOT.exists():
    shutil.rmtree(ROOT)
subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPOSITORY, str(ROOT)], check=True)
os.chdir(ROOT)
print('branch=', subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip())
print('commit=', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
github_token = secrets.get_secret('GITHUB_TOKEN')
if not github_token:
    raise RuntimeError('Kaggle Secret GITHUB_TOKEN is empty')
os.environ['GITHUB_TOKEN'] = github_token
try:
    hf_token = secrets.get_secret('HF_TOKEN')
except Exception:
    hf_token = None
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
print('GitHub secret loaded; HF token present=', bool(hf_token))

subprocess.run([
    sys.executable,
    'scripts/research/moe_mutation_001/publish.py',
    '--branch', BRANCH,
    '--preflight-only',
], check=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[lm,dev]'], check=True)

import torch
import transformers
import huggingface_hub
import safetensors

if not torch.cuda.is_available():
    raise RuntimeError('Formal MoE Mutation 001 requires CUDA')
print('torch=', torch.__version__)
print('transformers=', transformers.__version__)
print('huggingface_hub=', huggingface_hub.__version__)
print('safetensors=', safetensors.__version__)
print('cuda_device_count=', torch.cuda.device_count())
print('formal_device=', torch.cuda.get_device_name(0))
if torch.cuda.device_count() > 1:
    print('Additional GPUs are intentionally unused by the frozen protocol.')

In [ ]:
protocol_path = ROOT / 'research/validations/moe-mutation-001/protocol.json'
protocol = json.loads(protocol_path.read_text())
formal_seeds = [int(seed) for seed in protocol['formal_seeds']]
print('formal_seeds=', formal_seeds)
print('base_revision=', protocol['base']['revision'])
print('conversion_identity=', protocol['base']['conversion_manifest_identity_sha256'])

In [ ]:
published_root = ROOT / 'artifacts/experiments/moe-mutation-001'
for seed in formal_seeds:
    published_result = published_root / f'seed-{seed}' / 'result.json'
    if published_result.is_file():
        existing = json.loads(published_result.read_text())
        print(f'[seed={seed}] already published with status={existing.get("status")}; skipping')
        continue

    print(f'\n=== FORMAL SEED {seed} ===')
    subprocess.run([
        sys.executable,
        'scripts/research/moe_mutation_001/run_seed.py',
        '--seed', str(seed),
        '--device', 'cuda:0',
    ], check=True)

    result_path = ROOT / 'results/moe-mutation-001' / f'seed-{seed}' / 'result.json'
    result = json.loads(result_path.read_text())
    print(f'[seed={seed}] scientific_status={result["status"]}')

    subprocess.run([
        sys.executable,
        'scripts/research/moe_mutation_001/publish.py',
        '--seed', str(seed),
        '--branch', BRANCH,
    ], check=True)
    print(f'[seed={seed}] published before continuing')

In [ ]:
decision_path = ROOT / 'artifacts/experiments/moe-mutation-001/decision.json'
if decision_path.is_file():
    decision = json.loads(decision_path.read_text())
    print(json.dumps(decision, indent=2, sort_keys=True))
else:
    print('No aggregate decision exists yet.')
print('final_commit=', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())